# C3.2 Practice — Make Your HDB Model Better & More Trustworthy

**Module 3 · Machine Learning & GenAI — Coaching Add-on**

---

### The story so far
In C3.1 you built a model that predicts an HDB flat's resale price from **3 things**: floor area, lease year, and floor level. You deployed it in Streamlit. 

Today's question: **Is it any good? And can we make it better?**

### Your mission (about 1 hour)
1. **Measure** how wrong your current model is, in plain dollars.
2. **Add features** and see the error shrink.
3. **Compare two models** and learn how to pick one honestly.

> 💡 **How to use this notebook:** Run each cell with `Shift + Enter`. Cells marked **🔧 YOUR TURN** have blanks (`____`) for you to fill in. Stuck? Open the **▸ Hint** below it. Full answers are at the very bottom.

**Scenario:** You're a junior analyst at a property agency. Your manager says: *"Nice predictor — but how much can I trust the number it gives a client?"* Let's find out.

In [1]:
# Setup — run this first (needs internet to download the data)
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

url = ("https://raw.githubusercontent.com/flexfengfeng/6m-data-C3.2/"
       "main/notebooks/data/"
       "Resale_flat_prices_based_on_registration_date_from_Jan-2017_onwards.csv")
data = pd.read_csv(url)
data["floor_level"] = data["storey_range"].str.split(" ").str[0].astype(float)
print("Rows of real HDB sales loaded:", len(data))
data.head()

Rows of real HDB sales loaded: 233479


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,floor_level
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0,10.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0,1.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0,1.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0,4.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0,1.0


> 📊 **Reading the output:** the print line confirms how many real HDB resale records were downloaded (about 233,000 rows), and `data.head()` shows the first 5 rows so you can eyeball the columns. Check that `floor_level` appears as a number on the right — that's the column we just built from `storey_range` (e.g. `"10 TO 12"` → `10.0`). If you see the table, the data loaded correctly and you're ready for Part A.

In [3]:
print("Data types:")
print(data.dtypes)
print()
print("Missing values:")
missing = data.isna().sum()
print(f"missing num: {missing}")

print(missing[missing > 0])
print()
print("Numerical summary (excluding missing):")
print(data.describe(include="number").round(2))

Data types:
month                      str
town                       str
flat_type                  str
block                      str
street_name                str
storey_range               str
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
remaining_lease            str
resale_price           float64
floor_level            float64
dtype: object

Missing values:
missing num: month                  0
town                   0
flat_type              0
block                  0
street_name            0
storey_range           0
floor_area_sqm         0
flat_model             0
lease_commence_date    0
remaining_lease        0
resale_price           0
floor_level            0
dtype: int64
Series([], dtype: int64)

Numerical summary (excluding missing):
       floor_area_sqm  lease_commence_date  resale_price  floor_level
count       233479.00            233479.00     233479.00    233479.00
mean            96.69              1996.55     531028.

## 📏 Three scorecards, in plain English (no stats needed)

From here on we judge every model with three numbers. Here's what they really mean:

**MAE — "on average, how many dollars off?"**
Imagine the model guesses the price of 100 flats. For each one, you check how far the guess was from the true price (ignore whether it guessed too high or too low — just the size of the miss). Average those misses, and that's the MAE.
- MAE = S\$80,000 → "typically, the guess is about \$80k away from reality."
- **Lower is better.** It's in plain dollars, so anyone can understand it.

**MAPE — "on average, how many *percent* off?"**
Same idea as MAE, but each miss is measured *relative to that flat's price* instead of in raw dollars. A \$50k miss on a \$1.2M flat is small; the same \$50k miss on a \$300k flat is huge. MAPE captures that by turning every miss into a percentage, then averaging.
- MAPE = 15% → "typically, the guess is about 15% away from the true price."
- **Lower is better.** It's handy because it's fair across cheap and expensive flats, and easy to say out loud ("we're usually within ~15%").

**R² — "out of everything that makes prices differ, how much did the model figure out?"**
Flats sell for wildly different prices. Some of that is explainable (size, location, flat type…) and a good model should capture it. R² is the *fraction* of that price variation the model successfully explains, written as a number from 0 to 1:
- **R² = 0** → useless: no better than ignoring the flat and just guessing the average price every time.
- **R² = 0.70** → "the model accounts for about 70% of why flats differ in price." (The other 30% is stuff it can't see — renovations, the view, how keen the buyer was, luck.)
- **R² = 1.0** → perfect: every prediction dead-on (never happens in the real world).
- **Higher is better.**

**Why use all three?** MAE gives the error *in dollars you can feel*; MAPE gives it *as a percentage* that's fair across cheap and pricey flats; R² gives *what share of the puzzle* the model solved. Together they give the honest picture. As you add features and change models below, watch MAE and MAPE go **down** and R² go **up**.

## Part A — How wrong is the current (3-feature) model?

We'll rebuild your L05 model and, this time, **score** it.

**Key idea — train/test split:** We teach the model on 80% of the flats, then quietly test it on the other 20% it has *never seen*. That's the honest way to check it — like setting exam questions a student hasn't already memorised.

**Key idea — MAE (Mean Absolute Error):** the average gap between the predicted price and the real price. If MAE = \$60,000, the model is *on average* \$60k off. Lower is better.

**Key idea — MAPE (Mean Absolute Percentage Error):** the same average gap, but as a *percentage* of each flat's price. If MAPE = 15%, the model is *on average* 15% off — fair to compare across cheap and pricey flats. Lower is better.

**Key idea — R² (R-squared):** the share of the price variation the model explains, from 0 to 1. R² = 0 means it's no better than always guessing the average price; R² = 1 means perfect. Higher is better — it's a quick "how much of the picture did we capture?" score that complements the dollar/percentage views.

### ✋ Pause & Predict
Before running: with only 3 features, do you think the model will be off by closer to **\$20k**, **\$60k**, or **\$120k**? Write your guess, then run the cell.

In [5]:
# Part A — baseline model with the original 3 features
features_3 = ["floor_area_sqm", "lease_commence_date", "floor_level"]
X = data[features_3]
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae_baseline = mean_absolute_error(y_test, preds)
mape_baseline = mean_absolute_percentage_error(y_test, preds)
r2_baseline = r2_score(y_test, preds)
print(f"Baseline (3 features): on average off by S${mae_baseline:,.0f}")
print(f"Baseline (3 features): MAPE = {mape_baseline:.1%}  (average % off — fair across cheap & pricey flats)")
print(f"Baseline (3 features): R² = {r2_baseline:.3f}  (share of price variation explained, 1.0 = perfect)")

Baseline (3 features): on average off by S$102,499
Baseline (3 features): MAPE = 20.1%  (average % off — fair across cheap & pricey flats)
Baseline (3 features): R² = 0.500  (share of price variation explained, 1.0 = perfect)


> 📊 **Reading the output:** with just 3 features the model is on average **~S\$102,000 off (MAE)**, which is about **~20% off (MAPE)**, and its **R² is ~0.50** — it explains only about half of why prices differ between flats. That's a weak-but-not-useless starting score: clearly a lot of what drives price (location, flat type, …) is still missing. Write all three numbers down — every later part is measured against them.

**What do you notice?** That dollar figure is your starting score. Every change you make from here, you'll compare against it. Write it down.

## Part B — 🔧 YOUR TURN: Add more features

Right now the model knows nothing about **what kind of flat** it is or **where** it is. A 5-room in Bishan and a 2-room in Yishun of the same size get treated the same — no wonder it struggles.

We'll add two columns: `flat_type` and `town`. These are **labels**, not numbers, so we convert them into 0/1 columns with `pd.get_dummies()` (a flat either *is* "4 ROOM" or it isn't).

In [12]:
# Part B — add flat_type and town
numeric = ["floor_area_sqm", "lease_commence_date", "floor_level"]
category = ["flat_type", "town"]

# 🔧 YOUR TURN: build X_more by combining the numeric + category columns,
#    then turning the categories into 0/1 columns.
X_more = pd.get_dummies(data[numeric + category], columns=category)   # <- fill the blank
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_more, y, test_size=0.2, random_state=42)

richer = LinearRegression()
richer.fit(X_train, y_train)
richer_preds = richer.predict(X_test)
mae_richer = mean_absolute_error(y_test, richer_preds)
mape_richer = mean_absolute_percentage_error(y_test, richer_preds)
r2_richer = r2_score(y_test, richer_preds)

print(f"Baseline (3 features): MAE S${mae_baseline:,.0f}   MAPE {mape_baseline:.1%}   R² {r2_baseline:.3f}")
print(f"Richer  (5 features): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Improvement: S${mae_baseline - mae_richer:,.0f} smaller error, MAPE {mape_baseline - mape_richer:+.1%}, R² up {r2_richer - r2_baseline:+.3f}")

Baseline (3 features): MAE S$102,499   MAPE 20.1%   R² 0.500
Richer  (5 features): MAE S$82,761   MAPE 16.7%   R² 0.704
Improvement: S$19,738 smaller error, MAPE +3.4%, R² up +0.205


> 📊 **Reading the output:** adding `flat_type` and `town` drops the error from **~S\$102k to ~S\$83k (MAE)**, tightens **MAPE from ~20% to ~17%**, and lifts **R² from ~0.50 to ~0.70**. So telling the model *what kind* of flat it is and *where* it sits explains an extra ~20% of price variation — a big, cheap win. Same model (Linear Regression), just more relevant information. This is the single most important lesson of the lab: better features often beat fancier models.

<details><summary>▸ Hint</summary>

`get_dummies` needs to know which columns are categories. You already stored them in the variable `category`. So the blank is just `category`.
</details>

**What do you notice?** Did the error drop? By how much? More relevant information usually = better predictions — but not always, which is what Part C explores.

## Part B+ — 🚀 Enhanced features: can we push the error even lower?

Part B added `flat_type` and `town` and the error dropped nicely. But the raw HDB data still holds columns we haven't used:

- **`flat_model`** — e.g. "Improved", "Maisonette", "DBSS". The *design* of the flat, not just its room count.
- **`remaining_lease`** — how many years of lease are left. HDB prices fall as the lease runs down, so this is strongly price-related. It arrives as messy text (`"61 years 04 months"`) so we clean it into a single number of years.
- **`txn_year`** — the year the sale happened (pulled from the `month` column). Prices drift over time; this lets the model follow the market trend.

We add all three on top of the Part B features and re-score with the **same** Linear Regression, so any change is purely down to the features.

In [ ]:
# Part B+ — enhanced feature set (built on top of Part B's columns)

# 1) remaining_lease is text like "61 years 04 months" -> convert to a number of years
def lease_to_years(s):
    s = str(s)
    years = 0.0
    if "year" in s:
        years += float(s.split("year")[0].strip())
    if "month" in s:
        tail = s.split("year")[-1]
        digits = "".join(ch for ch in tail if ch.isdigit())
        if digits:
            years += float(digits) / 12
    return years

data["remaining_lease_years"] = data["remaining_lease"].apply(lease_to_years)

# 2) transaction year from the 'month' column ("2017-03" -> 2017)
data["txn_year"] = data["month"].str.split("-").str[0].astype(int)

numeric_plus  = ["floor_area_sqm", "lease_commence_date", "floor_level",
                 "remaining_lease_years", "txn_year"]
category_plus = ["flat_type", "town", "flat_model"]

X_plus = pd.get_dummies(data[numeric_plus + category_plus], columns=category_plus)
y = data["resale_price"]

X_train, X_test, y_train, y_test = train_test_split(X_plus, y, test_size=0.2, random_state=42)

enhanced = LinearRegression()
enhanced.fit(X_train, y_train)
enhanced_preds = enhanced.predict(X_test)
mae_plus  = mean_absolute_error(y_test, enhanced_preds)
mape_plus = mean_absolute_percentage_error(y_test, enhanced_preds)
r2_plus   = r2_score(y_test, enhanced_preds)

print(f"Baseline (3 features): MAE S${mae_baseline:,.0f}   MAPE {mape_baseline:.1%}   R\u00b2 {r2_baseline:.3f}")
print(f"Richer   (5 features): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R\u00b2 {r2_richer:.3f}")
print(f"Enhanced (8 features): MAE S${mae_plus:,.0f}   MAPE {mape_plus:.1%}   R\u00b2 {r2_plus:.3f}")
print()
print(f"Enhanced vs Richer:  S${mae_richer - mae_plus:,.0f} smaller error, "
      f"MAPE {mape_richer - mape_plus:+.1%}, R\u00b2 {r2_plus - r2_richer:+.3f}")

> 📊 **Reading the output (real numbers):** adding `remaining_lease_years`, `flat_model` and `txn_year` is a **big win** — the same Linear Regression jumps from Part B to:
>
> | Metric | Part B (5 feat) | Part B+ Enhanced (8 feat) | Change |
> |---|---|---|---|
> | **MAE**  | S\$82,761 | **S\$52,882** | ↓ S\$29,879 (~36% smaller) |
> | **MAPE** | 16.7% | **10.9%** | ↓ 5.8 points |
> | **R²**   | 0.704 | **0.868** | ↑ +0.164 |
>
> That's measured on all 233,479 real HDB records with the same 80/20 split. The standout driver is `remaining_lease_years` — lease decay is one of the strongest forces in HDB pricing, and Part B threw it away entirely. Note this **8-feature Linear Regression even beats the Random Forest in Part C** (~S\$73k MAE on the weaker feature set).

**The lesson, loud and clear:** before reaching for a fancier model (Part C), ask *"what relevant column am I still throwing away?"* Cleaning `remaining_lease` from text into a number is a tiny bit of work for a ~S\$30k accuracy gain.

**What do you notice?** Try removing `txn_year` or `flat_model` from the lists and re-running — does the error move much? That's a quick, honest way to see which features actually earn their place. (Hint: `remaining_lease_years` is the one doing most of the heavy lifting.)

## Part C — 🔧 YOUR TURN: Choose a model + check for overfitting

Linear Regression draws straight-line relationships. A **Random Forest** can capture curves and combinations (e.g. "top floor *in a central town* is worth extra"). Let's see if it does better.

**Watch out for overfitting:** a model can score brilliantly on flats it trained on but flop on new ones — like a student who memorised past papers but can't handle new questions. We catch this by comparing the **training score** with the **test score**. A big gap = memorising, not learning.

In [13]:
# Part C — compare Linear Regression vs Random Forest
# 🔧 YOUR TURN: create a Random Forest model in the blank.
forest = RandomForestRegressor(n_estimators=100, random_state=42)   # <- fill the blank
forest.fit(X_train, y_train)

# Test-set error (on unseen flats) vs training-set error (on flats it learned from)
mae_test   = mean_absolute_error(y_test,  forest.predict(X_test))
mae_train  = mean_absolute_error(y_train, forest.predict(X_train))
mape_test  = mean_absolute_percentage_error(y_test,  forest.predict(X_test))
mape_train = mean_absolute_percentage_error(y_train, forest.predict(X_train))
r2_test    = r2_score(y_test,  forest.predict(X_test))
r2_train   = r2_score(y_train, forest.predict(X_train))

print(f"Linear Regression  test: MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest      test: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print()
print(f"Random Forest  TRAIN: MAE S${mae_train:,.0f}   MAPE {mape_train:.1%}   R² {r2_train:.3f}")
print(f"Random Forest   TEST: MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gap (overfitting signal): MAE S${mae_test - mae_train:,.0f}, MAPE {mape_test - mape_train:+.1%}, R² {r2_train - r2_test:+.3f}")

Linear Regression  test: MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest      test: MAE S$72,629   MAPE 14.4%   R² 0.774

Random Forest  TRAIN: MAE S$62,456   MAPE 12.6%   R² 0.833
Random Forest   TEST: MAE S$72,629   MAPE 14.4%   R² 0.774
Gap (overfitting signal): MAE S$10,173, MAPE +1.8%, R² +0.059


> 📊 **Reading the output:** the Random Forest beats Linear Regression on the unseen test set — error falls to **~S\$73k (MAE)**, **~14% (MAPE)**, and **R² rises to ~0.77** — because it can capture curves and feature combinations a straight line can't. But look at the train-vs-test gap: it scores **better on flats it trained on** (MAE ~S\$62k, MAPE ~13%, R² ~0.83) than on new ones (MAE ~S\$73k, MAPE ~14%, R² ~0.77). That gap *is* overfitting — the model has partly memorised the training flats. The gap here is modest, so the forest is still the better model to ship; always judge by the **test** score, never the train score.

<details><summary>▸ Hint</summary>

We imported it at the top as `RandomForestRegressor`. Put that in the blank.
</details>

**What do you notice?**
- Which model has the lower **test** error? That's usually the one to ship.
- Is the Random Forest's **train** error much smaller than its **test** error? That gap is overfitting — it knows the training flats almost perfectly but is less sharp on new ones.

## Part C+ — 🔧 YOUR TURN:Enhanced  Try a *better* model — Gradient Boosting (and watch overfitting)

Random Forest grows ~100 trees **independently** and averages them. **Gradient Boosting** is different: it builds trees **one after another**, where each new tree focuses on fixing the *mistakes* the previous trees made. It's often the strongest model for tabular data like this — and `HistGradientBoostingRegressor` is a fast, modern version built into scikit-learn.

We'll judge it the same honest way as Part C: compare its **test** score against Random Forest, and check the **train-vs-test gap** to see how much it overfits.

> 🧠 **Key idea — the overfitting gap:** a model that scores *far* better on training flats than on unseen test flats has partly **memorised** the training data instead of learning general patterns. A *small* train↔test gap means the model generalises honestly. We want a low **test** error **and** a small gap.

In [ ]:
# Part C+ — Gradient Boosting vs Random Forest (uses the X_train/X_test already in scope)
from sklearn.ensemble import HistGradientBoostingRegressor

# 🔧 YOUR TURN: create a HistGradientBoostingRegressor in the blank.
booster = HistGradientBoostingRegressor(random_state=42)   # <- fill the blank
booster.fit(X_train, y_train)

def report(model, name):
    out = {}
    for split, Xs, ys in [("test", X_test, y_test), ("train", X_train, y_train)]:
        p = model.predict(Xs)
        out[split] = (mean_absolute_error(ys, p),
                      mean_absolute_percentage_error(ys, p),
                      r2_score(ys, p))
    print(f"{name:16s} TEST : MAE S${out['test'][0]:>8,.0f}   MAPE {out['test'][1]:5.1%}   R\u00b2 {out['test'][2]:.3f}")
    print(f"{name:16s} TRAIN: MAE S${out['train'][0]:>8,.0f}   MAPE {out['train'][1]:5.1%}   R\u00b2 {out['train'][2]:.3f}")
    gap = out['train'][2] - out['test'][2]
    print(f"{name:16s} overfitting gap (train R\u00b2 \u2212 test R\u00b2): {gap:+.3f}\n")
    return out

rf_out = report(forest,  "Random Forest")
gb_out = report(booster, "Gradient Boost")

better = "Gradient Boosting" if gb_out['test'][0] < rf_out['test'][0] else "Random Forest"
print(f"Lower TEST error -> ship: {better}")

> 📊 **Reading the output (real numbers on the 8-feature enhanced set):**
>
> | Model | TEST MAE | TEST R² | TRAIN R² | Overfitting gap |
> |---|---|---|---|---|
> | **Random Forest** | S\$25,769 | 0.962 | 0.994 | **+0.032** (large) |
> | **Gradient Boosting** | S\$32,499 | 0.946 | 0.947 | **+0.001** (tiny) |
>
> A genuinely instructive result — and *not* a clean "new model wins":
> - **Random Forest has the lower test error** (S\$25.8k vs S\$32.5k), so by the Part C rule — *ship the lowest test error* — it's still the one to deploy here.
> - **But look at the gaps.** Random Forest scores almost perfectly on flats it trained on (R² 0.994) yet drops to 0.962 on unseen ones — it has **memorised** a lot of the training set. Gradient Boosting's train and test scores are **basically identical** (0.947 vs 0.946) — it barely overfits at all. It's learning general patterns, not memorising.
>
> **The lesson:** "better" isn't only about the single lowest test number. A model that *doesn't* overfit is more trustworthy as data changes, and with a little tuning (more iterations, more depth via `max_iter` / `max_depth`) Gradient Boosting can usually close the gap to Random Forest while keeping its honesty. 

**What do you notice?**
- Which model would *you* ship — the lower-error one, or the one that generalises more honestly? (There's no single right answer; it depends on how much the data will drift.)
- Try `HistGradientBoostingRegressor(max_iter=500, learning_rate=0.1, random_state=42)` — can you beat Random Forest's test error *without* opening a big overfitting gap?

<details><summary>▸ Hint</summary>

We imported it as `HistGradientBoostingRegressor`. The blank is just `HistGradientBoostingRegressor(random_state=42)`.
</details>

### 🔧 Now tune it — can Gradient Boosting overtake Random Forest *without* overfitting?

The default Gradient Boosting lost to Random Forest on raw error. But its real strength is that it barely overfits — so we have room to make it **work harder** (more trees, a little more depth) and close the gap, while a touch of **regularisation** (`l2_regularization`) keeps it honest.

Knobs we'll turn:
- **`max_iter`** — how many boosting rounds (trees built in sequence). More = stronger, but slower and risks overfitting.
- **`learning_rate`** — how big a correction each new tree makes. Smaller + more trees usually generalises better.
- **`max_depth`** — how complex each individual tree can be.
- **`l2_regularization`** — a penalty that discourages the model from memorising.

In [ ]:
# Part C+ (tuned) — push Gradient Boosting and re-check the overfitting gap
tuned_gb = HistGradientBoostingRegressor(
    max_iter=500,            # more boosting rounds (default is 100)
    learning_rate=0.1,
    max_depth=8,
    l2_regularization=1.0,   # regularise -> resist memorising
    random_state=42,
)
tuned_gb.fit(X_train, y_train)

report(forest,    "Random Forest")
report(booster,   "GB (default)")
report(tuned_gb,  "GB (tuned)")

> 📊 **Reading the output (real numbers on the 8-feature enhanced set):**
>
> | Model | TEST MAE | TEST R² | TRAIN R² | Overfitting gap |
> |---|---|---|---|---|
> | Random Forest | **S\$25,769** | 0.962 | 0.994 | +0.032 (large) |
> | GB (default) | S\$32,499 | 0.946 | 0.947 | +0.001 (tiny) |
> | **GB (tuned)** | S\$26,439 | **0.964** | 0.966 | **+0.002** (tiny) |
>
> **This is the payoff.** Tuning transformed Gradient Boosting from the *worst* of the three into the **best generaliser**:
> - Its **test R² (0.964) now edges out Random Forest (0.962)** — the highest of all.
> - Its **overfitting gap stays tiny (+0.002)** vs Random Forest's large +0.032. Random Forest is still memorising; tuned GB is not.
> - On MAE the two are **essentially tied** (S\$26.4k vs S\$25.8k) — Random Forest is a hair lower, but it buys that with heavy overfitting.
>
> **Verdict:** the **tuned Gradient Boosting** is the model I'd ship — equal-or-better test scores *and* an honest train↔test gap. The cost is training time (~15s vs ~4s here) — a fine trade for a model you trust.

**What do you notice?**
- Did more `max_iter` help the **test** score, or just the train score? (Here it helped both, gap stayed tiny — that's the sign of healthy tuning.)
- Try removing `l2_regularization=1.0` or pushing `max_depth=15` — watch the gap widen. That's overfitting creeping back in.

### ✅ Evaluation & conclusion

Ranking the three models on the two things that matter *together* — **test accuracy** and **honesty (a small overfitting gap)**:

| Rank | Model | Why |
|---|---|---|
| 🥇 **1** | **GB (tuned)** | Highest test R² (0.964), MAE within ~S\$700 of the best, gap a tiny +0.002. Wins on *both* axes. |
| 🥈 2 | Random Forest | Lowest raw MAE (S\$25.8k) but a large +0.032 gap — it pays for accuracy by memorising. |
| 🥉 3 | GB (default) | Honest (gap +0.001) but under-powered until tuned (MAE S\$32.5k). |

**Conclusion:** tuning was the deciding move. The *same* `HistGradientBoostingRegressor` went from worst to best simply by working harder (`max_iter=500`) while staying disciplined (`l2_regularization=1.0`) — so it's the model to deploy. More broadly, the whole lab reveals the real order of leverage: **clean, relevant features first (the S\$102k → S\$53k drop), then a well-tuned model on top (S\$53k → S\$26k)** — and always judge by the *test* score paired with the *train↔test gap*, never accuracy alone.

## 🏆 Challenge (if you have time)
Pick **one**:
1. Add a third feature of your choice (e.g. `remaining_lease` cleaned to a number) and see if the error drops further.
2. Try `RandomForestRegressor(n_estimators=300)` — does more trees help the test error, or just slow it down?
3. Find the single flat in the test set where the model was **most wrong**. What was special about it?

## Part D — 🔧 (Stretch) Stacking: can a *team* of models beat the best one?

Your learner asked a great question: can we combine Random Forest and Gradient Boosting?

The feasible way to do this is **stacking** — train several different models *side by side*, then let one small final model (the "manager") learn how to best **blend** their predictions. The idea: each model makes different mistakes, so a smart blend can beat any single one.

> ⚠️ **Common misconception:** you do **not** chain them so Random Forest "narrows down" and Gradient Boosting "fine-tunes" the same price. Each model already predicts the full price on its own — you blend them, you don't hand one's answer to the other. (The one model that *does* "fine-tune its own mistakes step by step" is Gradient Boosting itself, internally.)

**Analogy:**
- Random Forest = average 100 independent guesses.
- Gradient Boosting = each guesser fixes the previous one's mistakes, in sequence.
- **Stacking** = collect guesses from several experts, then a manager learns whose opinion to trust when.

Let's test whether the extra complexity is actually worth it for our HDB data.

### ✋ Pause & Predict
The stacked team uses *more* computation than a single model. Do you think it will **beat** the best single model's test error, or only **tie** it for a lot more effort? Write your guess, then run.

In [6]:
# Part D — stack Random Forest + Gradient Boosting, blended by a Linear Regression "manager"
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor

# Two different base models (they make different kinds of mistakes)
base_models = [
    ("random_forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("grad_boost",    GradientBoostingRegressor(random_state=42)),
]

# 🔧 YOUR TURN: the "manager" that learns how to blend the base models.
#    Use a simple LinearRegression() as the final_estimator.
stack = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),          # <- fill the blank
)
stack.fit(X_train, y_train)
stack_preds = stack.predict(X_test)
mae_stack  = mean_absolute_error(y_test, stack_preds)
mape_stack = mean_absolute_percentage_error(y_test, stack_preds)
r2_stack   = r2_score(y_test, stack_preds)

# Also score Gradient Boosting on its own, for a fair comparison
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
mae_gb  = mean_absolute_error(y_test, gb_preds)
mape_gb = mean_absolute_percentage_error(y_test, gb_preds)
r2_gb   = r2_score(y_test, gb_preds)

print(f"Linear Regression (Part B): MAE S${mae_richer:,.0f}   MAPE {mape_richer:.1%}   R² {r2_richer:.3f}")
print(f"Random Forest    (Part C): MAE S${mae_test:,.0f}   MAPE {mape_test:.1%}   R² {r2_test:.3f}")
print(f"Gradient Boosting (alone) : MAE S${mae_gb:,.0f}   MAPE {mape_gb:.1%}   R² {r2_gb:.3f}")
print(f"Stacked team             : MAE S${mae_stack:,.0f}   MAPE {mape_stack:.1%}   R² {r2_stack:.3f}")

Linear Regression (Part B): MAE S$82,761   MAPE 16.7%   R² 0.704
Random Forest    (Part C): MAE S$72,629   MAPE 14.4%   R² 0.774
Gradient Boosting (alone) : MAE S$78,854   MAPE 15.7%   R² 0.725
Stacked team             : MAE S$71,903   MAPE 14.3%   R² 0.781


> 📊 **Reading the output:** Gradient Boosting alone lands around **~S\$79k (MAE), ~16% (MAPE), R² ~0.73**, and the **stacked team** comes in around **~S\$72k, ~14%, R² ~0.78** — essentially tied with the single Random Forest from Part C on all three scorecards. So blending three models bought us almost nothing here, despite ~3× the training work. That's the takeaway: stacking is a real competition-winning trick, but on data like this a single good model is usually within a whisker, so reach for the extra complexity only when that last sliver of accuracy genuinely pays off.

<details><summary>▸ Hint</summary>

The "manager" is just another model. We've used it all along: `LinearRegression()`. Put that in the blank.
</details>

**What do you notice?**
- Did the **stacked team** beat the best single model? By how many dollars?
- Was the improvement big enough to justify training *three* models instead of one?
- Often a single well-tuned Gradient Boosting model is within a whisker of the stack — which is why, in practice, you usually reach for stacking only when that last bit of accuracy really matters (it's a classic competition-winning trick, but frequently overkill).

## ✅ Wrap-up

| Idea | In one line |
|---|---|
| **MAE** | Average dollar error — your model's report card. |
| **Train/test split** | Test on unseen flats for an honest score. |
| **Adding features** | More relevant info usually shrinks the error. |
| **Model choice** | Pick the lowest **test** error, not training error. |
| **Overfitting** | Big train-vs-test gap = memorising, not learning. |

**Up next:** plug your better model into the upgraded `app.py` and redeploy in Streamlit.

---
## Sample solutions (look only after trying!)

**Part B blank:** `X_more = pd.get_dummies(data[numeric + category], columns=category)`

**Part C blank:** `forest = RandomForestRegressor(n_estimators=100, random_state=42)`


**Part D blank:** `final_estimator=LinearRegression()`